In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

Stick with regularization.

## Section 1: Augment Data

Purpose: Prevent overfitting/Improve Generalization

| Code | Description |
|---|---|
| `data_aug = data_nonaug + torch.randn_like(data_nonaug)*scale_noise_num` | Adds gaussian noise to the data. |
| `mask = torch.bernoulli(torch.ones_like(data_nonaug)*some_fraction)` <br> `data_aug = data_nonaug[mask]` | Randomly drops some fraction of the data. |


#### **Exercises**


Run the cell below to standardize the features in the wine dataset.

In [ ]:
wine.features = torch.tensor(StandardScaler().fit_transform(features_unscaled), dtype=torch.float32)
features_nonaug = wine.features

**Example**: Add gaussian noise to the features. Scale the noise by ``0.05`` (this makes the noise have a standard deviation of `0.05`).

In [ ]:
scale_noise_num = 0.05
features_aug = features_nonaug + torch.randn_like(features_nonaug)*scale_noise_num
features_aug

tensor([[ 1.5270, -0.5757,  0.1855,  ...,  0.3807,  1.7771,  1.0176],
        [ 0.3194, -0.5803, -0.7608,  ...,  0.4453,  1.1407,  1.0085],
        [ 0.1465, -0.0170,  1.0975,  ...,  0.2253,  0.7499,  1.3823],
        ...,
        [ 0.3790,  1.7315, -0.3516,  ..., -1.5757, -1.4178,  0.3039],
        [ 0.2433,  0.2029, -0.0117,  ..., -1.6011, -1.3347,  0.2123],
        [ 1.4286,  1.5688,  1.4352,  ..., -1.5230, -1.4506, -0.6622]])

**Exercise**: Add gaussian noise to the features. Scale the noise by ``0.7``. Display the features after the noise has been added.

In [ ]:
scale_noise_num = 0.7
features_aug = features_nonaug + torch.randn_like(features_nonaug)*scale_noise_num
features_aug

tensor([[ 1.6564, -0.6806,  0.3877,  ...,  0.5455,  2.1094, -0.1637],
        [-0.9906,  0.1093, -0.2197,  ..., -0.6490,  1.1086,  1.9808],
        [-0.2379,  1.4260,  1.8432,  ...,  0.5368, -0.4783,  1.5972],
        ...,
        [ 1.4691,  1.7774, -0.5627,  ..., -1.4946, -1.5475, -0.4870],
        [ 0.3266,  0.7279,  0.0239,  ..., -1.6136, -1.7619, -0.0483],
        [ 1.5144,  1.8035,  1.5755,  ..., -1.6144, -2.5515,  0.1073]])

**Exercise**: Drop 90 % of the features data.

In [ ]:
fraction_dropped = 0.9
features_aug = torch.bernoulli(torch.tensor(torch.ones_like(features_nonaug))*fraction_dropped)
features_aug

/tmp/ipykernel_314307/2107403781.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features_aug = torch.bernoulli(torch.tensor(torch.ones_like(features_nonaug))*fraction_dropped)


tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [0., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [0., 1., 0.,  ..., 1., 1., 1.]])

## Section 2: Regularize Model


| Code | Description |
|---|---|
| `for param in model.parameters():` | Loop through all parameters in `model`. |
| `l1 = 0.0`<br>`for param in model.parameters():`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`l1 += torch.sum(torch.abs(param))` | Calculate L1 norm for `model`. |
| `l2 = 0.0`<br>`for param in model.parameters():`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`l2 += torch.sum(param**2))` | Calculate L2 norm for `model`. |
| `loss = loss_function(pred, true) + lambda*l1` | L1 regularization: Add L1 penalty to loss function. |
| `loss = loss_function(pred, true) + lambda*l2` | L2 regularization: Add L2 penalty to loss function. |
| `nn.Dropout(0.1)` | Randomly removes 10 % of the neurons from layer. |




In [2]:
class Model(nn.Module):
    def __init__(self,):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(13, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16,13)
        )

    def forward(self, x):
        output = self.layers(x)
        return output
    


In [ ]:
model = Model()
np.sum(list(model.parameters()))

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [9]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.1004, -0.0902, -0.1811,  0.0395, -0.2117, -0.2189, -0.0960, -0.1123,
          -0.2738, -0.1315, -0.2158,  0.0474, -0.1116],
         [-0.1799,  0.0220,  0.1691, -0.1937, -0.0630,  0.1455, -0.2457, -0.0644,
          -0.0243, -0.1665, -0.1821,  0.0797, -0.1528],
         [-0.0989,  0.0899,  0.0816,  0.0237,  0.1163, -0.2572,  0.1265,  0.1262,
          -0.1958,  0.2473,  0.0909,  0.1054, -0.0223],
         [-0.2425,  0.0039,  0.1338,  0.2202,  0.1614, -0.0628,  0.0931, -0.1855,
           0.0344,  0.1690,  0.1766,  0.1317, -0.2584],
         [ 0.1444,  0.1348,  0.2346, -0.0103,  0.0626, -0.2205,  0.2534, -0.1447,
          -0.1242, -0.0198,  0.1006, -0.1000, -0.2561],
         [-0.0028, -0.2734, -0.1835,  0.1112,  0.0171, -0.0311,  0.2741,  0.2290,
          -0.2760, -0.1965,  0.2648,  0.2720,  0.1532],
         [-0.0312, -0.2384, -0.0648, -0.2142,  0.1554, -0.1409,  0.1032,  0.2677,
           0.0808,  0.2491, -0.1968,  0.0642,  0.0770],
         [ 

L1 Norm

In [ ]:
model = Model()
l1 = 0.0
for param in model.parameters():
  # param in one loop corresponds to all weights of each layer
  print(len(param))
  l1 += torch.sum(torch.abs(param))
l1

16
16
13
13


tensor(58.5785, grad_fn=<AddBackward0>)

L2 Norm

In [ ]:
model = Model()
l1 = 0.0
for param in model.parameters():
  # param in one loop corresponds to all weights of each layer
  print(len(param))
  l1 += torch.sum(param**2)
l1

In [ ]:
loss = loss_function(prediction, true_data) = l1